# Python 클래스 파일별 3 · 8.ipynb 상속 · 오버라이딩 · 다형성 · `super()`

**한 줄 요약** · Animal을 물려받은 Dog · Cat · Bird로 상속 · 오버라이딩 · 다형성 · `super()`를 확인하고, 같은 문법으로 `nn.Module` 모델을 만들어 연결

- **원본** · `2.ipynb/01_강의노트/8.ipynb` (원본 파일은 그대로 둠)
- **종합 정리본** · `00_최소선/02_Python클래스` 전체 정리 4장 · 5장

## 🎯 학습 목표

이 노트북을 끝내면 **자료를 보지 않고** 다음을 할 수 있어야 함

1. `class 자식(부모):`로 물려받고, 같은 이름의 메서드를 다시 정의(오버라이딩)한다
2. 같은 호출이 객체마다 다르게 동작하는 다형성을 예로 설명한다
3. `super()`로 부모 기능을 먼저 실행하고, 자식 `__init__` 첫 줄에 `super().__init__()`을 둔다
4. `nn.Module`을 물려받은 모델에서 `super().__init__()`을 빠뜨리면 생기는 에러를 말한다

## 원본 구성


- 원본 학습 목표 4개
  - 자식 클래스가 부모 메서드를 사용
  - 자식 클래스에서 부모 메서드를 다시 정의 (오버라이딩)
  - 같은 메서드가 객체에 따라 다르게 동작 (다형성)
  - `super()`로 부모 메서드 호출

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">원본 절</th>
<th style="text-align: left;">내용</th>
<th style="text-align: left;">이 노트북</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1 ~ 2</td>
<td style="text-align: left;">부모 클래스 · 그대로 상속</td>
<td style="text-align: left;">2-1</td>
</tr>
<tr>
<td style="text-align: left;">3 ~ 4</td>
<td style="text-align: left;">오버라이딩 · 다형성</td>
<td style="text-align: left;">2-2</td>
</tr>
<tr>
<td style="text-align: left;">5 ~ 6</td>
<td style="text-align: left;"><code>super()</code>로 메서드 확장 · 생성자에서 <code>super()</code></td>
<td style="text-align: left;">2-3</td>
</tr>
<tr>
<td style="text-align: left;">7</td>
<td style="text-align: left;">객체와 클래스의 관계 (<code>isinstance</code>)</td>
<td style="text-align: left;">2-4</td>
</tr>
<tr>
<td style="text-align: left;">보충</td>
<td style="text-align: left;"><code>nn.Module</code>에서 확인</td>
<td style="text-align: left;">2-5</td>
</tr>
</tbody>
</table>

- 원본 끝의 `dir()` 출력 셀(속성 목록 70여 개)은 제외 → 메서드를 찾는 순서(`__mro__`)로 대신 확인

---
## 1. 원리

> 🎯 **부모 클래스의 기능을 물려받아 일부만 바꾸고, 부모 기능 위에 덧붙이려면?**

💡 **쉽게 말하면**

- 상속은 **부모 레시피를 물려받아 쓰는 것**.
- 그대로 쓰면 상속, 한 단계를 바꿔 쓰면 오버라이딩, 부모 방식대로 먼저 하고 덧붙이면 `super()`.
- 파이썬은 메서드를 **자식 → 부모** 순서로 찾음.

<img src="images/cls2_inherit.png" alt="상속 관계와 메서드를 찾는 순서" width="763">

---
## 2. 풀이

단계마다 **먼저 예측** → 코드 실행 → **결과** 순서로 확인. 셀은 위에서부터 차례로 실행.

### 2-1. 부모 클래스와 그대로 상속 <sub>원본 1 ~ 2절</sub>

**실행 흐름**

1. Animal 정의
2. BasicDog(pass)
3. 두 객체의 메서드 호출

**먼저 예측** · 아무 메서드도 없는 `BasicDog`의 `sound()`는? — 답은 코드 실행 결과 아래에

In [1]:
class Animal:                  # 부모 클래스
    def __init__(self, name):  # 모든 동물의 이름
        self.name = name       # 자식도 물려받는 속성

    def introduce(self):                        # 자식이 그대로 쓸 메서드
        return f"제 이름은 {self.name}입니다."  # 이름으로 소개

    def sound(self):                    # 자식이 바꿀 메서드
        return "동물이 소리를 냅니다."  # 기본 소리


class BasicDog(Animal):  # 2절: 아무것도 쓰지 않고 물려받기
    pass


animal = Animal("동물")                          # 부모 객체
basic_dog = BasicDog("초코")                     # 자식 객체
print(animal.introduce(), animal.sound())        # 부모 메서드
print(basic_dog.introduce(), basic_dog.sound())  # 자식에 없으면 부모에서 찾음

제 이름은 동물입니다. 동물이 소리를 냅니다.
제 이름은 초코입니다. 동물이 소리를 냅니다.


**결과** · 예측 정답은 `동물이 소리를 냅니다.` — 부모 것을 그대로 씀

- `제 이름은 동물입니다. 동물이 소리를 냅니다.` — 부모 객체
- `제 이름은 초코입니다. 동물이 소리를 냅니다.` — BasicDog에 없으니 Animal에서 찾음

### 2-2. 오버라이딩과 다형성 <sub>원본 3 ~ 4절</sub>

**실행 흐름**

1. Dog · Cat이 sound() 다시 정의
2. dog의 두 메서드
3. 종류가 다른 객체 목록을 같은 호출로

**먼저 예측** · `[Animal, Dog, Cat]` 객체에 똑같이 `sound()`를 부르면? — 답은 코드 실행 결과 아래에

In [2]:
class Dog(Animal):     # Animal을 물려받음
    def sound(self):   # 같은 이름으로 다시 정의 (오버라이딩)
        return "멍멍"  # Dog만의 소리


class Cat(Animal):     # Animal을 물려받음
    def sound(self):   # 오버라이딩
        return "야옹"  # Cat만의 소리


dog, cat = Dog("콩이"), Cat("나비")  # 자식 객체 둘
print(dog.introduce(), dog.sound())  # introduce는 부모 것, sound는 Dog 것


# 4절 다형성: 같은 호출 sound() → 객체마다 다른 결과
animals = [Animal("동물"), Dog("초코"), Cat("나비")]         # 종류가 다른 객체
for current_animal in animals:                               # 종류를 묻지 않고 반복
    print(current_animal.name, "→", current_animal.sound())  # 같은 호출

제 이름은 콩이입니다. 멍멍
동물 → 동물이 소리를 냅니다.
초코 → 멍멍
나비 → 야옹


**결과** · 예측 정답은 객체마다 다른 소리 — 다형성

- `제 이름은 콩이입니다. 멍멍` — introduce는 물려받고 sound는 Dog 것
- `동물 → 동물이 소리를 냅니다.` · `초코 → 멍멍` · `나비 → 야옹` — 같은 `sound()` 호출, 객체마다 다른 결과

### 2-3. `super()`로 확장 · 생성자에서 `super()` <sub>원본 5 ~ 6절</sub>

**실행 흐름**

1. LoudDog: 부모 sound + !!!
2. Bird: `super().__init__` + can_fly
3. Bird 메서드 · `__dict__`

**먼저 예측** · `LoudDog("두부").sound()`는? — 답은 코드 실행 결과 아래에

In [3]:
class LoudDog(Dog):                     # Dog를 물려받음
    def sound(self):                    # 부모 sound를 확장
        parent_sound = super().sound()  # 부모(Dog)의 sound() → 멍멍
        return parent_sound + "!!!"     # 덧붙이기


class Bird(Animal):                     # Animal을 물려받음
    def __init__(self, name, can_fly):  # 자식은 인자가 하나 더
        super().__init__(name)          # name 저장은 부모에게 맡김
        self.can_fly = can_fly          # Bird에만 필요한 속성

    def sound(self):   # 오버라이딩
        return "짹짹"  # Bird만의 소리

    def show_flight(self):                          # 자식에만 있는 메서드
        if self.can_fly:                            # 날 수 있으면
            return f"{self.name}: 날 수 있습니다."  # 원본 문장
        return f"{self.name}: 날 수 없습니다."      # 아니면


print(LoudDog("두부").sound())                             # 부모 결과 + !!!
bird = Bird("참새", True)                                  # name은 부모, can_fly는 자식이 저장
print(bird.introduce(), bird.sound(), bird.show_flight())  # 물려받은 · 바꾼 · 새 메서드
print(bird.__dict__)                                       # 두 속성 모두 저장됨

멍멍!!!
제 이름은 참새입니다. 짹짹 참새: 날 수 있습니다.
{'name': '참새', 'can_fly': True}


**결과** · 예측 정답은 `멍멍!!!`

- `멍멍!!!` — `super().sound()`가 Dog의 멍멍을 가져와 덧붙임
- `제 이름은 참새입니다. 짹짹 참새: 날 수 있습니다.` — 물려받은 · 바꾼 · 새 메서드
- `{'name': '참새', 'can_fly': True}` — name은 부모 `__init__`이, can_fly는 자식이 저장

### 2-4. 객체와 클래스의 관계 · `super()`를 빠뜨리면 <sub>원본 7절 + 보충</sub>

**실행 흐름**

1. isinstance 3가지
2. 메서드를 찾는 순서
3. `super().__init__()` 생략한 자식

**먼저 예측** · `isinstance(dog, Animal)`은? — 답은 코드 실행 결과 아래에

In [4]:
print(isinstance(dog, Dog), isinstance(dog, Animal), isinstance(cat, Dog))  # 자식 = 부모 객체
print([c.__name__ for c in LoudDog.__mro__])                                # 메서드를 찾는 순서


class BrokenBird(Animal):               # super().__init__()을 빠뜨린 자식
    def __init__(self, name, can_fly):  # 부모 초기화 생략
        self.can_fly = can_fly          # name이 저장되지 않음


try:
    BrokenBird("펭귄", False).introduce()  # name을 쓰는 순간
except AttributeError as e:                # name 속성이 없음
    print("AttributeError:", e)            # 에러 문장

True True False
['LoudDog', 'Dog', 'Animal', 'object']
AttributeError: 'BrokenBird' object has no attribute 'name'


**결과** · 예측 정답은 `True` — Dog 객체는 Animal의 객체이기도 함

- `True True False` — dog는 Dog이자 Animal, cat은 Dog가 아님
- `['LoudDog', 'Dog', 'Animal', 'object']` — 자식 → 부모 → object 순서로 찾음
- `AttributeError: 'BrokenBird' object has no attribute 'name'` — 부모 초기화를 건너뛰어 name이 없음

### 2-5. `nn.Module`에서 확인 <sub>보충</sub>

**실행 흐름**

1. Net(nn.Module) · `super().__init__()`
2. forward 오버라이딩
3. `model(x)` · 물려받은 `parameters()`
4. super 생략한 BrokenNet

**먼저 예측** · `super().__init__()` 없이 `self.fc = nn.Linear(...)`를 쓰면 언제 에러가 날까? — 답은 코드 실행 결과 아래에

In [5]:
import torch          # 텐서
from torch import nn  # 층 · 모델


class Net(nn.Module):              # nn.Module을 물려받음
    def __init__(self):            # 층 준비
        super().__init__()         # 부모(nn.Module) 준비를 먼저
        self.fc = nn.Linear(4, 2)  # 층 등록

    def forward(self, x):  # 오버라이딩: 계산 방법만 정의
        return self.fc(x)  # 입력 4 → 점수 2


torch.manual_seed(0)                                                       # 가중치 초기값 고정
net = Net()                                                                # 모델 객체
# 상속 확인 · model(x) 출력 모양
print(isinstance(net, nn.Module), tuple(net(torch.ones(3, 4)).shape))
print("물려받은 parameters():", sum(p.numel() for p in net.parameters()))  # 4 × 2 + 2


class BrokenNet(nn.Module):        # super().__init__() 생략
    def __init__(self):            # 부모 준비 없이
        self.fc = nn.Linear(4, 2)  # 층 저장 → 즉시 에러


try:
    BrokenNet()                  # 만드는 순간
except AttributeError as e:      # 모듈을 등록할 공간이 없음
    print("AttributeError:", e)  # 에러 문장

True (3, 2)
물려받은 parameters(): 10
AttributeError: cannot assign module before Module.__init__() call


**결과** · 예측 정답은 만드는 순간 — 층을 저장하는 줄에서 바로 에러 (위 BrokenBird는 name을 쓸 때)

- `True (3, 2)` — Net은 nn.Module의 객체 · 배치 3 × 점수 2
- `물려받은 parameters(): 10` — 만들지 않은 메서드도 물려받음 · 4 × 2 + 2 = 10
- `AttributeError: cannot assign module before Module.__init__() call` — nn.Module은 super 생략을 즉시 알려 줌

---
## 3. 차이 · 언제 쓰나

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">그대로 상속 (<code>pass</code>)</td>
<td style="text-align: left;">부모 메서드를 <strong>그대로</strong> 사용</td>
<td style="text-align: left;">이름만 다르고 동작은 같을 때</td>
</tr>
<tr>
<td style="text-align: left;">오버라이딩</td>
<td style="text-align: left;">같은 이름으로 <strong>완전히 교체</strong></td>
<td style="text-align: left;">자식마다 동작이 달라야 할 때 (<code>forward</code>)</td>
</tr>
<tr>
<td style="text-align: left;"><code>super().메서드()</code></td>
<td style="text-align: left;">부모 결과를 받아 <strong>덧붙임</strong></td>
<td style="text-align: left;">부모 기능 + 추가 (<code>LoudDog</code>)</td>
</tr>
<tr>
<td style="text-align: left;"><code>super().__init__(...)</code></td>
<td style="text-align: left;">부모 속성 준비 <strong>후</strong> 자식 속성 추가</td>
<td style="text-align: left;">자식에 <code>__init__</code>이 있으면 <strong>항상</strong></td>
</tr>
<tr>
<td style="text-align: left;"><code>isinstance(x, 부모)</code></td>
<td style="text-align: left;">상속까지 인정 → True</td>
<td style="text-align: left;">자료형 확인 (<code>type(x) is</code>는 정확히 그 클래스만)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 자식에 `__init__`을 만들면 부모 `__init__`은 자동 실행 X → 첫 줄에 `super().__init__(...)`
> - `nn.Module`은 빠뜨리면 층을 저장하는 순간 바로 에러

> ✅ **한 줄 정리** · `class 자식(부모):`로 물려받아 필요한 메서드만 다시 정의하고, 부모 기능은 `super()`로 먼저 실행한 뒤 덧붙인다.

**딥러닝 코드에서는**

- `class Net(nn.Module):` → `super().__init__()` → `self.층` → `forward` 오버라이딩 (위 2-5)
- `for layer in layers: x = layer(x)` — 다형성 (`nn.Sequential`)

---
## 4. 확인 문제

아래 빈 셀에 **직접 작성** 후 실행. 다 푼 뒤에 모범 답안과 비교.

1. `Animal`을 물려받아 `sound()`가 `"음메"`인 `Cow`를 만들고 `introduce()` · `sound()` 출력
2. `LoudCat(Cat)`을 만들어 `super()`로 `"야옹?"`을 돌려주게 하기

In [6]:
# 직접 작성

**모범 답안** · 풀고 나서 보기 (각 print 오른쪽 주석이 실제 출력)

```python
class Cow(Animal):
    def sound(self):
        return "음메"


class LoudCat(Cat):
    def sound(self):
        return super().sound() + "?"


print(Cow("누렁이").introduce(), Cow("누렁이").sound())  # 제 이름은 누렁이입니다. 음메
print(LoudCat("나비").sound())  # 야옹?
```

---
## 다음에 볼 것

- 전체 정리 4장 · 5장 (PyTorch Dataset · nn.Module 종합 실습)
- 03_NumPy 전체 정리